# PaperTunedLLM — Qwen2.5-3B QLoRA Fine-tuning on QASPER


## 1. Setup


In [1]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

!rm -rf LLaMA-Factory
!git clone --depth 1 -q https://github.com/hiyouga/LLaMA-Factory.git
!cd LLaMA-Factory && pip install -q -e .
!cd LLaMA-Factory && pip install -q -r requirements/metrics.txt

!pip install -q --no-deps -U "peft>=0.20.0"
!pip install -q --no-deps -U "torchao>=0.16.0"
!pip install -q --no-deps -U "bitsandbytes>=0.46.1"
!pip install -q --no-deps -U "datasets>=3.4.1,<4.4.0,!=4.0.*,!=4.1.0"

!pip list | grep -E "^(torchao|peft|trl|accelerate|bitsandbytes|datasets|transformers|unsloth) "


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 48.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 102.7 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 98.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 92.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 2. Dataset Preparation


In [2]:
%%writefile prepare_qasper.py

import json
import os
from datasets import load_dataset


def extract_answer(answer_group):
    if isinstance(answer_group, dict):
        annotations = answer_group.get("answer", [])
    elif isinstance(answer_group, list):
        annotations = answer_group
    else:
        annotations = []

    for answer in annotations:
        if not isinstance(answer, dict):
            continue
        if answer.get("unanswerable", False):
            continue

        free_form_answer = answer.get("free_form_answer")
        if free_form_answer:
            return free_form_answer.strip()

        extractive_spans = answer.get("extractive_spans") or []
        if extractive_spans:
            return ", ".join(str(span).strip() for span in extractive_spans if span)

        yes_no = answer.get("yes_no")
        if yes_no is True:
            return "Yes"
        elif yes_no is False and (
            answer.get("free_form_answer") is not None
            or answer.get("extractive_spans") is not None
        ):
            return "No"

    return ""


def build_full_text(item):
    full_text = item.get("full_text") or {}
    section_names = full_text.get("section_name") or []
    paragraphs_by_section = full_text.get("paragraphs") or []

    sections = []
    for section_name, paragraphs in zip(section_names, paragraphs_by_section):
        section_text = "\n".join(str(p).strip() for p in paragraphs if p).strip()
        if section_text:
            heading = section_name.strip() if section_name else "Section"
            sections.append(f"### {heading}\n{section_text}")

    return "\n\n".join(sections)


def prepare_qasper_for_llamafactory(output_path: str):
    dataset = load_dataset("allenai/qasper", revision="refs/convert/parquet", split="train")

    formatted_data = []
    for item in dataset:
        title = item.get("title", "")
        abstract = item.get("abstract", "")
        full_text = build_full_text(item)

        qas = item.get("qas") or {}
        questions = qas.get("question") or []
        answers = qas.get("answers") or []

        for question, answer_group in zip(questions, answers):
            answer_text = extract_answer(answer_group)
            if not answer_text:
                continue

            context = (
                f"Title: {title}\n\n"
                f"Abstract: {abstract}\n\n"
                f"Full paper text:\n{full_text}"
            ).strip()

            formatted_data.append({
                "instruction": "Answer the following question based on the provided scientific research paper.",
                "input": f"Question: {question.strip()}\n\nContext:\n{context}",
                "output": answer_text,
            })

    output_dir = os.path.dirname(output_path)
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(formatted_data, f, indent=2, ensure_ascii=False)

    print(f"Saved {len(formatted_data)} examples to: {output_path}")


if __name__ == "__main__":
    prepare_qasper_for_llamafactory("qasper_llamafactory.json")


Writing prepare_qasper.py


In [3]:
!python prepare_qasper.py
!cp qasper_llamafactory.json LLaMA-Factory/data/
!jq '. + {"qasper": {"file_name": "qasper_llamafactory.json"}}' LLaMA-Factory/data/dataset_info.json > temp.json && mv temp.json LLaMA-Factory/data/dataset_info.json


qasper/train/0000.parquet: 100%|███████████| 14.4M/14.4M [00:00<00:00, 16.9MB/s]
qasper/validation/0000.parquet: 100%|██████| 4.75M/4.75M [00:01<00:00, 4.69MB/s]
qasper/test/0000.parquet: 100%|████████████| 7.07M/7.07M [00:00<00:00, 17.1MB/s]
Generating train split: 888 examples [00:00, 4905.50 examples/s]
Generating validation split: 281 examples [00:00, 7709.56 examples/s]
Generating test split: 416 examples [00:00, 7164.48 examples/s]
Saved 2322 examples to: qasper_llamafactory.json


## 3. Training Config


In [4]:
%%writefile llamafactory_qlora.yaml
### model
model_name_or_path: Qwen/Qwen2.5-3B
quantization_bit: 4
use_unsloth: true

### method
stage: sft
do_train: true
finetuning_type: lora
lora_target: all

### dataset
dataset: qasper
template: qwen
cutoff_len: 1024
max_samples: 1000
overwrite_cache: true
preprocessing_num_workers: 16

### output
output_dir: saves/Qwen2.5-3B/lora/sft
logging_steps: 10
save_steps: 100
plot_loss: true
overwrite_output_dir: true

### train
per_device_train_batch_size: 4
gradient_accumulation_steps: 4
learning_rate: 2.0e-4
num_train_epochs: 2.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
fp16: true
ddp_timeout: 180000000


Writing llamafactory_qlora.yaml


## 4. Train


In [5]:
!cd LLaMA-Factory && DISABLE_VERSION_CHECK=1 CUDA_VISIBLE_DEVICES=0 llamafactory-cli train ../llamafactory_qlora.yaml


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
[WARNING|2026-08-06 10:01:37] llamafactory.extras.misc:155 >> Version checking has been disabled, may lead to unexpected behaviors.
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape sequence '\s'
  re_skip_default = re.compile("(\r\n|\s)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/finalseg/__init__.py:78: SyntaxWarning: invalid escape sequence '\.'
  re_skip = re.compile("([a-zA-Z0-9]+(?:\.\d+)?%?)")
/usr/local/lib/python3.12/dist-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 202

## 5. Inference


In [6]:
%%writefile llamafactory_infer.yaml
model_name_or_path: Qwen/Qwen2.5-3B
adapter_name_or_path: saves/Qwen2.5-3B/lora/sft
template: qwen
finetuning_type: lora


Writing llamafactory_infer.yaml


In [3]:
!cd LLaMA-Factory &&DISABLE_VERSION_CHECK=1 CUDA_VISIBLE_DEVICES=0 llamafactory-cli chat ../llamafactory_infer.yaml

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
[WARNING|2026-08-06 11:33:26] llamafactory.extras.misc:155 >> Version checking has been disabled, may lead to unexpected behaviors.
[INFO|configuration_utils.py:765] 2026-08-06 11:33:27,220 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-3B/snapshots/3aab1f1954e9cc14eb9509a215f9e5ca08227a9b/config.json
[INFO|configuration_utils.py:841] 2026-08-06 11:33:27,225 >> Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151643,
  "hidden_act": "silu",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size": 11008,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_atten

## 6. Export (merge adapter into base weights)


In [1]:
%%writefile llamafactory_export.yaml
model_name_or_path: Qwen/Qwen2.5-3B
adapter_name_or_path: saves/Qwen2.5-3B/lora/sft
template: qwen
finetuning_type: lora
export_dir: saves/Qwen2.5-3B/merged
export_size: 2
export_device: cpu
export_legacy_format: false


Overwriting llamafactory_export.yaml


In [2]:
!cd LLaMA-Factory &&DISABLE_VERSION_CHECK=1 llamafactory-cli export ../llamafactory_export.yaml

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
[WARNING|2026-08-06 11:37:13] llamafactory.extras.misc:155 >> Version checking has been disabled, may lead to unexpected behaviors.
/usr/local/lib/python3.12/dist-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[INFO|configuration_utils.py:765] 2026-08-06 11:37:17,829 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-3B/snapshots/3aab1f1954e9cc14eb9509a215f9e5ca08227a9b/config.json
[INFO|configuration_utils.py:841] 2026-08-06 11:37:17,834 >> Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151

## 7. Push to Hugging Face Hub


In [3]:
from huggingface_hub import login
login()


In [4]:
from huggingface_hub import HfApi

HF_USERNAME = "ranjeet258"
REPO_NAME = "PaperTunedLLM-Qwen2.5-3B-QASPER"
repo_id = f"{HF_USERNAME}/{REPO_NAME}"

api = HfApi()
api.create_repo(repo_id=repo_id, exist_ok=True, private=True)
api.upload_folder(
    folder_path="LLaMA-Factory/saves/Qwen2.5-3B/lora/sft",
    repo_id=repo_id,
    commit_message="PaperTunedLLM LoRA adapter",
)
print(f"https://huggingface.co/{repo_id}")


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

https://huggingface.co/ranjeet258/PaperTunedLLM-Qwen2.5-3B-QASPER


## 8. Download Locally


In [5]:
import shutil

shutil.make_archive("/kaggle/working/papertunedllm_adapter", "zip", "LLaMA-Factory/saves/Qwen2.5-3B/lora/sft")
print("Ready: /kaggle/working/papertunedllm_adapter.zip")


Ready: /kaggle/working/papertunedllm_adapter.zip


In [6]:
%%writefile prepare_qasper_val.py

import json
import os
from datasets import load_dataset

from prepare_qasper import extract_answer, build_full_text


def prepare_qasper_val(output_path: str, max_examples: int = 200):
    dataset = load_dataset("allenai/qasper", revision="refs/convert/parquet", split="validation")

    formatted_data = []
    for item in dataset:
        if len(formatted_data) >= max_examples:
            break

        title = item.get("title", "")
        abstract = item.get("abstract", "")
        full_text = build_full_text(item)

        qas = item.get("qas") or {}
        questions = qas.get("question") or []
        answers = qas.get("answers") or []

        for question, answer_group in zip(questions, answers):
            if len(formatted_data) >= max_examples:
                break

            answer_text = extract_answer(answer_group)
            if not answer_text:
                continue

            context = (
                f"Title: {title}\n\n"
                f"Abstract: {abstract}\n\n"
                f"Full paper text:\n{full_text}"
            ).strip()

            formatted_data.append({
                "instruction": "Answer the following question based on the provided scientific research paper.",
                "input": f"Question: {question.strip()}\n\nContext:\n{context}",
                "output": answer_text,
            })

    output_dir = os.path.dirname(output_path)
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(formatted_data, f, indent=2, ensure_ascii=False)

    print(f"Saved {len(formatted_data)} validation examples to: {output_path}")


if __name__ == "__main__":
    prepare_qasper_val("qasper_val_llamafactory.json", max_examples=200)


Writing prepare_qasper_val.py


In [7]:
!python prepare_qasper_val.py
!cp qasper_val_llamafactory.json LLaMA-Factory/data/
!jq '. + {"qasper_val": {"file_name": "qasper_val_llamafactory.json"}}' LLaMA-Factory/data/dataset_info.json > temp.json && mv temp.json LLaMA-Factory/data/dataset_info.json


Saved 200 validation examples to: qasper_val_llamafactory.json


In [8]:
%%writefile llamafactory_eval_finetuned.yaml
model_name_or_path: Qwen/Qwen2.5-3B
adapter_name_or_path: saves/Qwen2.5-3B/lora/sft
template: qwen
finetuning_type: lora

stage: sft
do_predict: true
predict_with_generate: true

eval_dataset: qasper_val
cutoff_len: 1024
max_samples: 200

output_dir: saves/eval/finetuned
per_device_eval_batch_size: 4
fp16: true


Writing llamafactory_eval_finetuned.yaml


In [17]:
%%writefile llamafactory_eval_base.yaml
model_name_or_path: Qwen/Qwen2.5-3B
template: qwen
finetuning_type: lora

stage: sft
do_predict: true
predict_with_generate: true

eval_dataset: qasper_val
cutoff_len: 1024
max_samples: 200

output_dir: saves/eval/base
per_device_eval_batch_size: 4
fp16: true


Overwriting llamafactory_eval_base.yaml


In [19]:
!python -c "from prepare_qasper_val import prepare_qasper_val; prepare_qasper_val('qasper_val_llamafactory.json', max_examples=50)"

Saved 50 validation examples to: qasper_val_llamafactory.json


In [22]:
!cp qasper_val_llamafactory.json LLaMA-Factory/data/

In [23]:
!cd LLaMA-Factory && DISABLE_VERSION_CHECK=1 CUDA_VISIBLE_DEVICES=0 llamafactory-cli train ../llamafactory_eval_finetuned.yaml


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
[WARNING|2026-08-06 12:14:07] llamafactory.extras.misc:155 >> Version checking has been disabled, may lead to unexpected behaviors.
/usr/local/lib/python3.12/dist-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[INFO|2026-08-06 12:14:11] llamafactory.hparams.parser:614 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: torch.float16
[INFO|configuration_utils.py:765] 2026-08-06 12:14:11,757 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-3B/snapshots/3aab1f1954e9cc14eb9509a215f9e5ca08227a9b/config.json
[INFO|configuratio

In [24]:
!cd LLaMA-Factory && DISABLE_VERSION_CHECK=1 CUDA_VISIBLE_DEVICES=0 llamafactory-cli train ../llamafactory_eval_base.yaml


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
[WARNING|2026-08-06 12:51:38] llamafactory.extras.misc:155 >> Version checking has been disabled, may lead to unexpected behaviors.
/usr/local/lib/python3.12/dist-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[INFO|2026-08-06 12:51:43] llamafactory.hparams.parser:614 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: torch.float16
[INFO|configuration_utils.py:765] 2026-08-06 12:51:43,426 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-3B/snapshots/3aab1f1954e9cc14eb9509a215f9e5ca08227a9b/config.json
[INFO|configuratio

In [26]:
import json

with open("LLaMA-Factory/saves/eval/finetuned/predict_results.json") as f:
    finetuned_metrics = json.load(f)

with open("LLaMA-Factory/saves/eval/base/predict_results.json") as f:
    base_metrics = json.load(f)

metric_keys = ["predict_rouge-1", "predict_rouge-2", "predict_rouge-l", "predict_bleu-4"]

print(f"{'Metric':<18}{'Base':>10}{'Fine-tuned':>14}{'Change':>10}")
for k in metric_keys:
    base_val = base_metrics.get(k, 0)
    ft_val = finetuned_metrics.get(k, 0)
    print(f"{k:<18}{base_val:>10.3f}{ft_val:>14.3f}{ft_val - base_val:>+10.3f}")

Metric                  Base    Fine-tuned    Change
predict_rouge-1        5.138        10.924    +5.786
predict_rouge-2        0.802         2.398    +1.596
predict_rouge-l        1.342         0.925    -0.417
predict_bleu-4         1.200         1.054    -0.146


In [27]:
# Side-by-side sample answers for manual inspection
import json

def load_predictions(path, n=5):
    rows = []
    with open(path) as f:
        for line in f:
            rows.append(json.loads(line))
            if len(rows) >= n:
                break
    return rows

base_preds = load_predictions("LLaMA-Factory/saves/eval/base/generated_predictions.jsonl")
finetuned_preds = load_predictions("LLaMA-Factory/saves/eval/finetuned/generated_predictions.jsonl")

for i, (b, ft) in enumerate(zip(base_preds, finetuned_preds)):
    print(f"--- Example {i+1} ---")
    print("Reference :", ft["label"][:200])
    print("Base      :", b["predict"][:200])
    print("Fine-tuned:", ft["predict"][:200])
    print()


--- Example 1 ---
Reference : BIBREF19, BIBREF20

Base      : - We study the zero-shot translation scenario for transfer learning in NMT and identify the domain shift problem as the root cause of the failure in zero-shot translation.
- We design a novel cross-li
Fine-tuned: The best reported results on twoNo need for pivot languages, better than the best results of BIBREF8 on both test sets, the best reported results of BIBREF9 on the test set, the best reported results 

--- Example 2 ---
Reference : pivoting, pivoting$_{\rm m}$

Base      :  propose a novel transfer learning approach for zero-shot translation based on cross-lingual pre-training, which has better performance than strong pivot-based baseline and various multilingual NMT ap
Fine-tuned: Two models trained separately, BIBREF8, BIBREF9, and BIBREF12. BRLM is compared with two models trained separately, BIBREF8, BIBREF9, and BIBREF12. BRLM is also compared with a model that is trained b

--- Example 3 ---
Reference : Europa

In [28]:
import subprocess
print(subprocess.run(["du", "-sh", "LLaMA-Factory/saves/Qwen2.5-3B/merged"], capture_output=True, text=True).stdout)

5.8G	LLaMA-Factory/saves/Qwen2.5-3B/merged



In [37]:
import os

def print_tree(start_path="/kaggle", indent=""):
    items = sorted(os.listdir(start_path))
    for i, item in enumerate(items):
        path = os.path.join(start_path, item)
        is_last = (i == len(items) - 1)
        print(indent + ("└── " if is_last else "├── ") + item)
        if os.path.isdir(path):
            print_tree(path, indent + ("    " if is_last else "│   "))

print_tree("/kaggle")

├── input
├── lib
│   └── kaggle
│       └── gcp.py
└── working
    ├── .virtual_documents
    │   └── __notebook_source__.ipynb
    ├── LLaMA-Factory
    │   ├── .ai
    │   │   └── CLAUDE.md
    │   ├── .claude
    │   │   └── skills
    │   │       └── llamafactory-sft
    │   │           └── SKILL.md
    │   ├── .dockerignore
    │   ├── .env.local
    │   ├── .git
    │   │   ├── HEAD
    │   │   ├── branches
    │   │   ├── config
    │   │   ├── description
    │   │   ├── hooks
    │   │   │   ├── applypatch-msg.sample
    │   │   │   ├── commit-msg.sample
    │   │   │   ├── fsmonitor-watchman.sample
    │   │   │   ├── post-update.sample
    │   │   │   ├── pre-applypatch.sample
    │   │   │   ├── pre-commit.sample
    │   │   │   ├── pre-merge-commit.sample
    │   │   │   ├── pre-push.sample
    │   │   │   ├── pre-rebase.sample
    │   │   │   ├── pre-receive.sample
    │   │   │   ├── prepare-commit-msg.sample
    │   │   │   ├── push-to-checkout.sample
    │   │   │   └

In [51]:
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

{'status': 'ok', 'restart': True}

/usr/local/lib/python3.12/dist-packages/IPython/core/history.py:576: DeprecationWarning: The default datetime adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  self.db.execute("""UPDATE sessions SET end=?, num_cmds=? WHERE


In [48]:
ls /kaggle/working/LLaMA-Factory/saves/Qwen2.5-3B/merged

chat_template.jinja               model-00004-of-00004.safetensors
config.json                       Modelfile
generation_config.json            model.safetensors.index.json
model-00001-of-00004.safetensors  tokenizer_config.json
model-00002-of-00004.safetensors  tokenizer.json
model-00003-of-00004.safetensors


/usr/lib/python3.12/pty.py:95: DeprecationWarning: This process (pid=789) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


In [1]:
from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer

# Point this to the merged model directory from your notebook
model_path = "LLaMA-Factory/saves/Qwen2.5-3B/merged"
quant_path = "LLaMA-Factory/saves/Qwen2.5-3B/awq-4bit"

# Configure AWQ settings
quant_config = {
    "zero_point": True,
    "q_group_size": 128,
    "w_bit": 4,
    "version": "GEMM"
}

print("Loading model and tokenizer...")
model = AutoAWQForCausalLM.from_pretrained(model_path, safetensors=True)
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)

print("Quantizing the model (this may take a bit)...")
# The tokenizer is passed so AutoAWQ can automatically calibrate
model.quantize(tokenizer, quant_config=quant_config)

print("Saving the quantized model...")
model.save_quantized(quant_path, safetensors=True, shard_size="4GB")
tokenizer.save_pretrained(quant_path)

print(f"Success! Your AWQ model is ready at: {quant_path}")

/usr/local/lib/python3.12/dist-packages/awq/__init__.py:21: DeprecationWarning: 
I have left this message as the final dev message to help you transition.

Important Notice:
- AutoAWQ is officially deprecated and will no longer be maintained.
- The last tested configuration used Torch 2.6.0 and Transformers 4.51.3.
- If future versions of Transformers break AutoAWQ compatibility, please report the issue to the Transformers project.

Alternative:
- AutoAWQ has been adopted by the vLLM Project: https://github.com/vllm-project/llm-compressor

For further inquiries, feel free to reach out:
- X: https://x.com/casper_hansen_
- LinkedIn: https://www.linkedin.com/in/casper-hansen-804005170/

  warnings.warn(_FINAL_DEV_MESSAGE, category=DeprecationWarning, stacklevel=1)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes

Loading model and tokenizer...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Quantizing the model (this may take a bit)...


...

Generating validation split:   0%|          | 0/214670 [00:00<?, ? examples/s]

AWQ: 100%|██████████| 36/36 [51:46<00:00, 86.29s/it]

Saving the quantized model...


Writing model shards: 0it [00:00, ?it/s]

Success! Your AWQ model is ready at: LLaMA-Factory/saves/Qwen2.5-3B/awq-4bit


In [3]:
import shutil
from IPython.display import FileLink

# 1. Zip the AWQ model directory
zip_path = "/kaggle/working/qwen2_5_3b_awq_4bit"
shutil.make_archive(zip_path, "zip", "LLaMA-Factory/saves/Qwen2.5-3B/awq-4bit")

print("Zip archive created successfully!")

# 2. Generate a direct download link in notebook
FileLink("qwen2_5_3b_awq_4bit.zip")

Zip archive created successfully!


/kaggle/working/qwen2_5_3b_awq_4bit.zip

In [4]:
from huggingface_hub import HfApi

HF_USERNAME = "ranjeet258"
REPO_NAME = "Qwen2.5-3B-QASPER-AWQ-4bit"
repo_id = f"{HF_USERNAME}/{REPO_NAME}"

api = HfApi()

# Create repo if it doesn't exist
api.create_repo(repo_id=repo_id, exist_ok=True, private=True)

# Upload the quantized folder
api.upload_folder(
    folder_path="LLaMA-Factory/saves/Qwen2.5-3B/awq-4bit",
    repo_id=repo_id,
    commit_message="Upload 4-bit AWQ quantized model",
)

print(f"Model successfully uploaded to: https://huggingface.co/{repo_id}")

INFO:httpx:HTTP Request: POST https://huggingface.co/api/repos/create "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://huggingface.co/api/models/ranjeet258/Qwen2.5-3B-QASPER-AWQ-4bit/preupload/main "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://huggingface.co/ranjeet258/Qwen2.5-3B-QASPER-AWQ-4bit.git/info/lfs/objects/batch "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/ranjeet258/Qwen2.5-3B-QASPER-AWQ-4bit/xet-write-token/main "HTTP/1.1 200 OK"


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

INFO:httpx:HTTP Request: POST https://huggingface.co/api/models/ranjeet258/Qwen2.5-3B-QASPER-AWQ-4bit/commit/main "HTTP/1.1 200 OK"


Model successfully uploaded to: https://huggingface.co/ranjeet258/Qwen2.5-3B-QASPER-AWQ-4bit
